<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/8ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_ranking.py

from __future__ import annotations

import json
from dataclasses import dataclass, field
from datetime import datetime
from enum import IntEnum, verify, UNIQUE
from pathlib import Path
from typing import Any, Final

from modul_filtry import (
    StatusFiltra,
    TypFiltra,
    WynikFiltra,
)

from modul_skaner import (
    WynikSkanera,
)


FOLDER_PROJEKTU: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)


@verify(UNIQUE)
class PoziomJakosci(IntEnum):
    BARDZO_NISKI = 1
    NISKI = 2
    SREDNI = 3
    WYSOKI = 4
    BARDZO_WYSOKI = 5


@dataclass(
    frozen=True,
    slots=True,
    order=True,
    kw_only=True,
)
class PozycjaRankingu:

    sort_index: float = field(
        init=False,
        repr=False,
        compare=True,
    )

    wynik_punktowy: float = field(
        compare=False,
    )

    ticker: str = field(
        compare=False,
    )

    pozycja: int = field(
        default=0,
        compare=False,
    )

    poziom: PoziomJakosci = field(
        default=PoziomJakosci.SREDNI,
        compare=False,
    )

    liczba_passed: int = field(
        default=0,
        compare=False,
    )

    liczba_filtrow: int = field(
        default=0,
        compare=False,
    )

    timestamp: datetime = field(
        default_factory=datetime.now,
        compare=False,
        hash=False,
        repr=False,
    )

    def __post_init__(self) -> None:

        object.__setattr__(
            self,
            "sort_index",
            -self.wynik_punktowy,
        )


@dataclass(
    slots=True,
    kw_only=True,
)
class Ranking:

    pozycje: list[PozycjaRankingu] = field(
        default_factory=list,
    )

    liczba_spolek: int = field(
        init=False,
    )

    najlepszy_ticker: str | None = field(
        init=False,
        default=None,
    )

    def __post_init__(self) -> None:

        self.pozycje.sort()

        self.liczba_spolek = len(
            self.pozycje
        )

        if self.pozycje:

            self.najlepszy_ticker = (
                self.pozycje[0].ticker
            )

        nowe_pozycje: list[
            PozycjaRankingu
        ] = []

        for numer, pozycja in enumerate(
            self.pozycje,
            start=1,
        ):

            nowa = PozycjaRankingu(
                ticker=pozycja.ticker,
                wynik_punktowy=(
                    pozycja.wynik_punktowy
                ),
                pozycja=numer,
                poziom=pozycja.poziom,
                liczba_passed=(
                    pozycja.liczba_passed
                ),
                liczba_filtrow=(
                    pozycja.liczba_filtrow
                ),
                timestamp=(
                    pozycja.timestamp
                ),
            )

            nowe_pozycje.append(
                nowa
            )

        self.pozycje = nowe_pozycje


class KalkulatorRankingu:

    def oblicz_punkty(
        self,
        wynik: WynikSkanera,
    ) -> float:

        if not wynik.przeszedl:

            raise ValueError(
                f"spolka {wynik.ticker} "
                "nie przeszla skanowania"
            )

        punkty: float = 0.0

        for filtr in wynik.wyniki_filtrow:

            if (
                filtr.status
                != StatusFiltra.PASSED
            ):
                continue

            punkty += 10.0

            if (
                filtr.typ_filtra
                == TypFiltra.SMA
            ):

                if filtr.prog != 0:

                    przewaga: float = (
                        filtr.wartosc
                        - filtr.prog
                    )

                    punkty += max(
                        0.0,
                        min(
                            przewaga * 100.0,
                            20.0,
                        ),
                    )

            elif (
                filtr.typ_filtra
                == TypFiltra.MOMENTUM
            ):

                punkty += max(
                    0.0,
                    min(
                        filtr.wartosc,
                        25.0,
                    ),
                )

            elif (
                filtr.typ_filtra
                == TypFiltra.ADVANCED_MOMENTUM
            ):

                punkty += max(
                    0.0,
                    min(
                        filtr.wartosc,
                        25.0,
                    ),
                )

            elif (
                filtr.typ_filtra
                == TypFiltra.ZMIENNOSC
            ):

                roznica: float = (
                    filtr.prog
                    - filtr.wartosc
                )

                punkty += max(
                    0.0,
                    min(
                        roznica * 2.0,
                        20.0,
                    ),
                )

        return round(
            punkty,
            2,
        )

    def okresl_poziom(
        self,
        punkty: float,
    ) -> PoziomJakosci:

        if punkty >= 80:
            return (
                PoziomJakosci.BARDZO_WYSOKI
            )

        if punkty >= 60:
            return (
                PoziomJakosci.WYSOKI
            )

        if punkty >= 40:
            return (
                PoziomJakosci.SREDNI
            )

        if punkty >= 20:
            return (
                PoziomJakosci.NISKI
            )

        return (
            PoziomJakosci.BARDZO_NISKI
        )


class RankingService:

    def __init__(
        self,
        kalkulator: KalkulatorRankingu,
    ) -> None:

        self.kalkulator = kalkulator

    def utworz_ranking(
        self,
        wyniki: list[WynikSkanera],
    ) -> Ranking:

        pozycje: list[
            PozycjaRankingu
        ] = []

        for wynik in wyniki:

            if not wynik.przeszedl:
                continue

            punkty: float = (
                self.kalkulator
                .oblicz_punkty(
                    wynik
                )
            )

            poziom: PoziomJakosci = (
                self.kalkulator
                .okresl_poziom(
                    punkty
                )
            )

            pozycja = PozycjaRankingu(
                ticker=wynik.ticker,
                wynik_punktowy=punkty,
                poziom=poziom,
                liczba_passed=(
                    wynik.liczba_passed
                ),
                liczba_filtrow=(
                    wynik.liczba_filtrow
                ),
            )

            pozycje.append(
                pozycja
            )

        return Ranking(
            pozycje=pozycje
        )


def wczytaj_wynik_skanera(
    ticker: str,
    folder: Path = FOLDER_PROJEKTU,
) -> WynikSkanera:

    ticker = (
        ticker
        .strip()
        .upper()
    )

    plik: Path = (
        folder
        / f"{ticker}_skaner.json"
    )

    if not plik.exists():

        raise FileNotFoundError(
            f"brak pliku z modulu 7: "
            f"{plik}"
        )

    with open(
        plik,
        "r",
        encoding="utf-8",
    ) as f:

        dane: Any = json.load(
            f
        )

    if not isinstance(
        dane,
        dict,
    ):

        raise ValueError(
            "dane skanera musza "
            "byc obiektem JSON"
        )

    if "ticker" not in dane:
        raise ValueError(
            "brak ticker"
        )

    if "przeszedl" not in dane:
        raise ValueError(
            "brak pola przeszedl"
        )

    if "wyniki_filtrow" not in dane:
        raise ValueError(
            "brak wyniki_filtrow"
        )

    wyniki_filtrow: list[
        WynikFiltra
    ] = []

    for rekord in dane[
        "wyniki_filtrow"
    ]:

        if not isinstance(
            rekord,
            dict,
        ):
            raise ValueError(
                "wynik filtra musi "
                "byc slownikiem"
            )

        try:

            typ_filtra = TypFiltra(
                rekord["typ"]
            )

        except ValueError as e:

            raise ValueError(
                f"nieznany typ filtra: "
                f"{rekord['typ']}"
            ) from e

        try:

            status = StatusFiltra(
                rekord["status"]
            )

        except ValueError as e:

            raise ValueError(
                f"nieznany status filtra: "
                f"{rekord['status']}"
            ) from e

        wynik_filtra = WynikFiltra(
            typ_filtra=typ_filtra,
            status=status,
            wartosc=float(
                rekord["wartosc"]
            ),
            prog=float(
                rekord["prog"]
            ),
            opis=str(
                rekord.get(
                    "opis",
                    "",
                )
            ),
        )

        wyniki_filtrow.append(
            wynik_filtra
        )

    wynik = WynikSkanera(
        ticker=str(
            dane["ticker"]
        ).strip().upper(),

        przeszedl=bool(
            dane["przeszedl"]
        ),

        wyniki_filtrow=(
            wyniki_filtrow
        ),
    )

    return wynik


def ranking_do_dict(
    ranking: Ranking,
) -> dict[str, Any]:

    return {

        "liczba_spolek":
            ranking.liczba_spolek,

        "najlepszy_ticker":
            ranking.najlepszy_ticker,

        "ranking": [

            {
                "pozycja":
                    pozycja.pozycja,

                "ticker":
                    pozycja.ticker,

                "wynik_punktowy":
                    pozycja.wynik_punktowy,

                "poziom": {
                    "nazwa":
                        pozycja.poziom.name,

                    "wartosc":
                        pozycja.poziom.value,
                },

                "liczba_passed":
                    pozycja.liczba_passed,

                "liczba_filtrow":
                    pozycja.liczba_filtrow,

                "timestamp":
                    pozycja.timestamp.isoformat(),
            }

            for pozycja
            in ranking.pozycje
        ],
    }


def zapisz_ranking(
    ranking: Ranking,
    folder: Path = FOLDER_PROJEKTU,
) -> Path:

    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    plik: Path = (
        folder
        / "ranking.json"
    )

    dane: dict[str, Any] = (
        ranking_do_dict(
            ranking
        )
    )

    with open(
        plik,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            dane,
            f,
            ensure_ascii=False,
            indent=2,
        )

    return plik


def zapisz_pozycje_spolek(
    ranking: Ranking,
    folder: Path = FOLDER_PROJEKTU,
) -> list[Path]:

    zapisane: list[Path] = []

    for pozycja in ranking.pozycje:

        plik: Path = (
            folder
            / f"{pozycja.ticker}_ranking.json"
        )

        dane: dict[str, Any] = {

            "ticker":
                pozycja.ticker,

            "pozycja":
                pozycja.pozycja,

            "wynik_punktowy":
                pozycja.wynik_punktowy,

            "poziom": {
                "nazwa":
                    pozycja.poziom.name,

                "wartosc":
                    pozycja.poziom.value,
            },

            "liczba_passed":
                pozycja.liczba_passed,

            "liczba_filtrow":
                pozycja.liczba_filtrow,

            "timestamp":
                pozycja.timestamp.isoformat(),
        }

        with open(
            plik,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                dane,
                f,
                ensure_ascii=False,
                indent=2,
            )

        zapisane.append(
            plik
        )

    return zapisane


def run() -> None:

    tekst: str = input(
        "podaj tickery oddzielone "
        "przecinkami: "
    )

    tickery: list[str] = [

        ticker.strip().upper()

        for ticker
        in tekst.split(",")

        if ticker.strip()
    ]

    if not tickery:

        raise ValueError(
            "nie podano tickerow"
        )

    print(
        "\nwczytywanie wynikow "
        "z modulu 7..."
    )

    wyniki_skanowania: list[
        WynikSkanera
    ] = []

    for ticker in tickery:

        wynik: WynikSkanera = (
            wczytaj_wynik_skanera(
                ticker
            )
        )

        wyniki_skanowania.append(
            wynik
        )

        print(
            ticker,
            "->",
            (
                "PASSED"
                if wynik.przeszedl
                else "FAILED"
            )
        )

    kalkulator = (
        KalkulatorRankingu()
    )

    ranking_service = (
        RankingService(
            kalkulator=kalkulator
        )
    )

    ranking: Ranking = (
        ranking_service.utworz_ranking(
            wyniki_skanowania
        )
    )

    print(
        "\nRANKING"
    )

    if not ranking.pozycje:

        print(
            "brak spolek, ktore "
            "przeszly skanowanie"
        )

    else:

        for pozycja in ranking.pozycje:

            print(
                pozycja.pozycja,
                ".",
                pozycja.ticker,
                "| punkty:",
                pozycja.wynik_punktowy,
                "| poziom:",
                pozycja.poziom.name,
                "| enum:",
                pozycja.poziom.value,
            )

    plik_rankingu: Path = (
        zapisz_ranking(
            ranking
        )
    )

    pliki_spolek: list[Path] = (
        zapisz_pozycje_spolek(
            ranking
        )
    )

    print(
        "\nzapisano ranking:"
    )

    print(
        plik_rankingu
    )

    for plik in pliki_spolek:

        print(
            plik
        )

    print(
        "\nMODUL RANKINGU "
        "DZIALA POPRAWNIE"
    )